# Cognitive TabICLv2 + Logistic Regression weighted hybrid

Development-only post hoc exploratory evaluation. This notebook reads the aggregate report only; it does not load participant data, models, contexts, or the previously examined test cohort.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
report = json.loads((root / 'reports/generated/development-cognitive-tabiclv2-weighted-hybrid-report.json').read_text(encoding='utf-8'))
assert report['status'] == 'completed_development_only_exploratory_weighted_hybrid'
assert report['test_cohort_loaded'] is False
assert report['development_participants'] == 500
assert report['weights'] == [0.0, 0.25, 0.5, 0.75, 1.0]
pd.Series({'status': report['status'], 'participants': report['development_participants'], 'outer folds': report['cross_validation']['outer_folds_completed'], 'runtime seconds': report['runtime_seconds']})

status             completed_development_only_exploratory_weighte...
participants                                                     500
outer folds                                                       25
runtime seconds                                           949.258514
dtype: object

## Participant-averaged out-of-fold metrics

In [2]:
rows = []
for name, method in report['methods'].items():
    m = method['participant_averaged_oof_metrics']
    rows.append({'Method': name, 'PR-AUC': m['average_precision_pr_auc'], 'AUROC': m['roc_auc'], 'Accuracy': m['accuracy'], 'Balanced accuracy': m['balanced_accuracy'], 'Sensitivity': m['sensitivity'], 'Specificity': m['specificity'], 'PPV': m['precision'], 'NPV': m['negative_predictive_value'], 'F1': m['f1'], 'Brier score': m['brier_score']})
pd.DataFrame(rows).set_index('Method').round(4)

,PR-AUC,AUROC,Accuracy,Balanced accuracy,Sensitivity,Specificity,PPV,NPV,F1,Brier score
Method,,,,,,,,,,
logistic,0.5604,0.8884,0.766,0.8276,0.9057,0.7494,0.3000,0.9853,0.4507,0.0656
tabiclv2,0.5802,0.9060,0.806,0.8333,0.8679,0.7987,0.3382,0.9808,0.4868,0.0605
equal_weight,0.5835,0.9030,0.794,0.8349,0.8868,0.7830,0.3264,0.9831,0.4772,0.0620
selected_weight,0.5842,0.9035,0.808,0.8427,0.8868,0.7987,0.3431,0.9835,0.4947,0.0621


## Paired PR-AUC uncertainty

In [3]:
intervals = []
for comparison, value in report['paired_bootstrap_pr_auc'].items():
    intervals.append({'Comparison': comparison, 'Point difference': value['point_difference'], 'Bootstrap mean difference': value['bootstrap_mean_difference'], 'Lower 95%': value['confidence_interval_95'][0], 'Upper 95%': value['confidence_interval_95'][1]})
pd.DataFrame(intervals).set_index('Comparison').round(4)

,Point difference,Bootstrap mean difference,Lower 95%,Upper 95%
Comparison,,,,
weighted_minus_tabiclv2,0.0040,0.0032,-0.0346,0.0411
weighted_minus_logistic,0.0238,0.0281,-0.0110,0.0679
weighted_minus_equal_weight,0.0007,0.0009,-0.0131,0.0161


## Weight and threshold stability

In [4]:
weights = pd.DataFrame({'weight': list(report['weight_selection_frequency'].keys()), 'folds selected': list(report['weight_selection_frequency'].values()), 'proportion': list(report['weight_selection_proportion'].values())})
display(weights)
pd.DataFrame(report['threshold_variability']).T.round(4)

,weight,folds selected,proportion
0,0.0,2,0.08
1,0.25,6,0.24
2,0.5,8,0.32
3,0.75,6,0.24
4,1.0,3,0.12


,mean,standard_deviation,minimum,maximum
logistic,0.1061,0.0274,0.0561,0.1791
tabiclv2,0.1171,0.0475,0.0598,0.2048
equal_weight,0.1202,0.0393,0.0695,0.1964
selected_weight,0.1204,0.0361,0.0621,0.2163


## Interpretation

The predefined weighted grid was selected inside each outer-training partition. The selected-weight result is compared with each individual benchmark and fixed 50:50 voting through participant-level paired bootstrap intervals. This post hoc development-only analysis cannot establish independent validation, clinical diagnostic accuracy, or a deployment decision. A numerical difference whose interval includes zero is not reliable evidence of superiority.